In [1]:
import numpy 
import torch 
import torch.nn as nn 
from torch.utils.data import TensorDataset, DataLoader

In [8]:
class GRU(nn.Module):
    def __init__(self,input_size,hidden_size):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size,hidden_size=hidden_size,batch_first=True)
        self.fc = nn.Linear(hidden_size,2)

    def forward(self,X):
        output,hn = self.gru(X)
        last_hidden = hn[-1]
        logits = self.fc(last_hidden)
        return logits

In [9]:
def create_dataset(num_samples, seq_length=10, input_size=5):
    X = torch.randn(num_samples, seq_length, input_size)
    y = (X.sum(dim=(1, 2)) > 0).long()

    return X, y


X_train, y_train = create_dataset(1000)
X_test, y_test = create_dataset(200)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True
)

test_loader = DataLoader(
    test_dataset, batch_size=32
)

In [10]:
model = GRU(input_size=5, hidden_size=16)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for e in range(20):

    model.train()

    total_loss = 0
    correct = 0 
    total = 0 

    for X,y in train_loader:

        logits = model(X)

        loss = criterion(logits,y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)


        predictions = logits.argmax(dim=1)
        correct += (predictions == y).sum().item()
        total += y.size(0)

    train_loss = total_loss/total
    train_accuracy = correct/total
    print(
        f"Epoch [{e + 1:02d}/{e}] "
        f"Loss: {train_loss:.4f} "
        f"Accuracy: {train_accuracy:.4f}"
    )

    model.eval()

    correct = 0 
    total = 0 
    test_loss = 0 

    with torch.no_grad():

        for Xt,yt in test_loader:
            logits = model(Xt)

            loss = criterion(logits,yt)

            test_loss+=loss.item() * Xt.size(0)

            predictions = logits.argmax(dim=1)

            correct += (predictions == yt).sum().item()
            total+= yt.size(0)

        test_accuracy = correct/total 
        total_loss = test_loss/total
    print("\nTest Results")
    print(f"Test Loss: {total_loss:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")



Epoch [01/0] Loss: 0.6911 Accuracy: 0.5190

Test Results
Test Loss: 0.6770
Test Accuracy: 0.5850
Epoch [02/1] Loss: 0.6610 Accuracy: 0.6590

Test Results
Test Loss: 0.6460
Test Accuracy: 0.7300
Epoch [03/2] Loss: 0.6136 Accuracy: 0.7460

Test Results
Test Loss: 0.5802
Test Accuracy: 0.7800
Epoch [04/3] Loss: 0.4769 Accuracy: 0.8620

Test Results
Test Loss: 0.3592
Test Accuracy: 0.9100
Epoch [05/4] Loss: 0.2826 Accuracy: 0.9040

Test Results
Test Loss: 0.2462
Test Accuracy: 0.9200
Epoch [06/5] Loss: 0.2180 Accuracy: 0.9320

Test Results
Test Loss: 0.2120
Test Accuracy: 0.9300
Epoch [07/6] Loss: 0.1824 Accuracy: 0.9380

Test Results
Test Loss: 0.1955
Test Accuracy: 0.9350
Epoch [08/7] Loss: 0.1615 Accuracy: 0.9530

Test Results
Test Loss: 0.1846
Test Accuracy: 0.9350
Epoch [09/8] Loss: 0.1474 Accuracy: 0.9500

Test Results
Test Loss: 0.1738
Test Accuracy: 0.9300
Epoch [10/9] Loss: 0.1363 Accuracy: 0.9610

Test Results
Test Loss: 0.1740
Test Accuracy: 0.9150
Epoch [11/10] Loss: 0.1279 Acc